# SPAM DETECTOR - MODÉLISATION ET ENTRAÎNEMENT

Ce notebook est entièrement dédié à la phase de Machine Learning. À ce stade, nos données textuelles ont été nettoyées, tokenisées et converties en représentations numériques (scores TF-IDF). Nous manipulons désormais uniquement des matrices et des vecteurs mathématiques.

## Importation des dépendances et préparation de l'environnement

Pour cette phase de modélisation, nous importons toutes les bibliothèques requises :
* `NumPy` : Pour charger et manipuler nos matrices numériques vectorisées.
* `Scikit-Learn (sklearn)` : Pour diviser le jeu de données, instancier nos différents modèles de référence (Modèles baselines) et générer les rapports d'évaluation (matrices de confusion, scores de performance).

In [13]:
import numpy as np
from sklearn.model_selection import train_test_split
import joblib
import pickle

## Chargement des matrices vectorisées et Séparation des données (Train/Test Split)

Dans cette section, nous importons la matrice de caractéristiques `X_features` et le vecteur cible `y_labels` que nous avons sérialisés sous format binaire NumPy (`.npy`) à la fin de l'étape de vectorisation.

Ensuite, nous procédons à la séparation des données en deux sous-ensembles :
* **Jeu d'entraînement (Train set - 80%)** : Données utilisées pour ajuster les poids et paramètres des modèles.
* **Jeu de test (Test set - 20%)** : Données indépendantes gardées secrètes, indispensables pour évaluer de manière impartiale l'efficacité opérationnelle des algorithmes face à des messages inédits.

Nous utilisons l'argument `stratify=y_labels` pour garantir que la proportion de spams et de hams soit rigoureusement identique dans les deux ensembles.

In [14]:
# Chargement des structures de données sauvegardées
X_features = np.load('../src/X_features.npy')
y_labels = np.load('../src/y_labels.npy')

print(f"Dimensions de la matrice globale X_features : {X_features.shape}")
print(f"Dimensions du vecteur global y_labels      : {y_labels.shape}\n")

# Séparation des données (80% entraînement, 20% test)
# L'argument random_state=42 assure la reproductibilité exacte du mélange
X_train, X_test, y_train, y_test = train_test_split(
    X_features, 
    y_labels, 
    test_size=0.2, 
    random_state=42, 
    stratify=y_labels
)

# Validation des dimensions des sous-ensembles
print(f"--- Répartition des données après découpage ---")
print(f"Sous-ensemble d'entraînement (X_train) : {X_train.shape[0]} lignes")
print(f"Sous-ensemble d'évaluation   (X_test)  : {X_test.shape[0]} lignes\n")

# Vérification de l'équilibre des classes dans le jeu d'entraînement
unique_train, counts_train = np.unique(y_train, return_counts=True)
print("Proportions dans le Train set :")
for label, count in zip(unique_train, counts_train):
    class_name = "Ham" if label == 0 else "Spam"
    print(f"  - {class_name} : {count} messages ({count/len(y_train)*100:.2f}%)")

Dimensions de la matrice globale X_features : (1306, 4185)
Dimensions du vecteur global y_labels      : (1306,)

--- Répartition des données après découpage ---
Sous-ensemble d'entraînement (X_train) : 1044 lignes
Sous-ensemble d'évaluation   (X_test)  : 262 lignes

Proportions dans le Train set :
  - Ham : 522 messages (50.00%)
  - Spam : 522 messages (50.00%)


## Implémentation Mathématique du Modèle: La Régression Logistique *From Scratch*

Après analyse de notre phase comparative, nous avons choisi de développer entièrement notre propre classifieur de spams basé sur l'algorithme de la **Régression Logistique** à l'aide de la bibliothèque `NumPy`. 

Bien qu'il s'agisse d'un problème de classification binaire (0 = Ham, 1 = Spam), le cœur de cet algorithme repose sur une approche de modélisation continue, transformée en probabilité par une fonction d'activation non linéaire.

---

### A. Fondations Mathématiques

#### La Combinaison Linéaire ($z$)
Pour chaque message du jeu de données (représenté par son vecteur de scores TF-IDF $X$), le modèle calcule d'abord une combinaison linéaire. Il associe à chaque caractéristique (mot du vocabulaire) un poids d'importance ($W$) et ajoute un terme de biais global ($b$) :

$$z = X \cdot W + b = \sum_{i=1}^{n} w_i x_i + b$$

* Un poids $w_i$ fortement **positif** indique un terme fortement corrélé aux spams (*"free"*, *"txt"*, *"claim"*).
* Un poids $w_i$ **négatif ou nul** indique un mot neutre ou caractéristique des messages légitimes (*"ham"*).

#### L'Activation Non-Linéaire : La Fonction Sigmoïde ($\sigma$)
Le score linéaire brut $z$ pouvant s'étendre de $-\infty$ à $+\infty$, il est indispensable de le projeter dans un intervalle $[0, 1]$ afin de l'interpréter comme une probabilité stricte. Pour cela, on applique la fonction **Sigmoïde** :

$$\hat{y} = \sigma(z) = \frac{1}{1 + e^{-z}}$$

* $\hat{y}$ représente la probabilité mathématique prédite par le modèle que le message soit un spam ($P(y=1 \mid X)$).

---

### B. Optimisation et Apprentissage : La Descente de Gradient

Pour ajuster les paramètres $W$ et $b$ au fil des époques, le modèle doit mesurer son erreur globale et la minimiser.

#### Fonction de Coût : L'Entropie Croisée Binaire (*Binary Cross-Entropy / Log Loss*)
Au lieu d'une simple erreur quadratique, on utilise la *Log Loss*, qui pénalise de manière logarithmique (et très sévèrement) les prédictions confiantes mais erronées :

$$J(W, b) = - \frac{1}{m} \sum_{i=1}^{m} \left[ y^{(i)} \log(\hat{y}^{(i)}) + (1 - y^{(i)}) \log(1 - \hat{y}^{(i)}) \right]$$

Où $m$ est le nombre total d'échantillons d'entraînement, $y$ la vraie étiquette (0 ou 1) et $\hat{y}$ la valeur prédite.

#### Calcul des Gradients (Dérivées Partielles)
Pour savoir comment modifier nos poids afin de réduire la perte $J$, on calcule la dérivée de la fonction de coût par rapport à $W$ et à $b$ en appliquant la règle de dérivation en chaîne (*Chain Rule*) :

$$\frac{\partial J}{\partial W} = \frac{1}{m} X^T (\hat{Y} - Y)$$

$$\frac{\partial J}{\partial b} = \frac{1}{m} \sum_{i=1}^{m} (\hat{y}^{(i)} - y^{(i)})$$

#### Règle de Mise à Jour des Paramètres
À chaque itération (époque), les poids et le biais sont mis à jour dans le sens inverse du gradient (pour descendre vers le minimum de la fonction de perte), modulés par le taux d'apprentissage ($\alpha$, ou *Learning Rate*) :

$$W \leftarrow W - \alpha \frac{\partial J}{\partial W}$$

$$b \leftarrow b - \alpha \frac{\partial J}{\partial b}$$

---

### C. Algorithme Général de notre Classe Python
L'architecture de notre classe `LogisticRegressionFromScratch` se structure autour de 3 méthodes principales :
1. `_sigmoid(z)` : Calcule l'activation en sécurisant les valeurs contre les débordements d'exposants (*overflow* via un `np.clip`).
2. `fit(X, y)` : Initialise les poids à zéro et exécute la boucle d'entraînement sur un nombre fixé d'époques pour optimiser $W$ et $b$.
3. `predict(X, threshold=0.5)` : Calcule la probabilité d'un message et applique un seuil strict (0.5) pour renvoyer la classe finale ($0$ ou $1$).

In [15]:
class LogisticRegressionFromScratch:
    def __init__(self, learning_rate=0.01, epochs=1000):
        self.lr = learning_rate
        self.epochs = epochs
        self.weights = None
        self.bias = None
        self.loss_history = []

    def _sigmoid(self, z):
        # Utilisation de np.clip pour éviter les débordements d'exposant (overflow)
        z = np.clip(z, -500, 500)
        return 1 / (1 + np.exp(-z))

    def fit(self, X, y):
        n_samples, n_features = X.shape
        
        # Initialisation des paramètres à zéro
        self.weights = np.zeros(n_features)
        self.bias = 0.0

        # Boucle d'entraînement (Descente de gradient)
        for epoch in range(self.epochs):
            # Combinaison linéaire : z = X * W + b
            linear_model = np.dot(X, self.weights) + self.bias
            
            # Activation non-linéaire : y_predicted = sigmoide(z)
            y_predicted = self._sigmoid(linear_model)

            # Calcul de la fonction de coût (Binary Cross-Entropy / Log Loss)
            # Ajout d'un epsilon pour éviter log(0)
            epsilon = 1e-15
            y_predicted = np.clip(y_predicted, epsilon, 1 - epsilon)
            loss = - (1 / n_samples) * np.sum(y * np.log(y_predicted) + (1 - y) * np.log(1 - y_predicted))
            self.loss_history.append(loss)

            # Calcul des Gradients
            dw = (1 / n_samples) * np.dot(X.T, (y_predicted - y))
            db = (1 / n_samples) * np.sum(y_predicted - y)

            # Mise à jour des poids et du biais
            self.weights -= self.lr * dw
            self.bias -= self.lr * db

            # Affichage de la progression toutes les 100 époques
            if epoch % 100 == 0:
                print(f"Époque {epoch}/{self.epochs} - Perte (Loss): {loss:.4f}")

    def predict_proba(self, X):
        linear_model = np.dot(X, self.weights) + self.bias
        return self._sigmoid(linear_model)

    def predict(self, X, threshold=0.5):
        probas = self.predict_proba(X)
        return np.where(probas >= threshold, 1, 0)

In [16]:
#  Instanciation du modèle avec les hyperparamètres
custom_model = LogisticRegressionFromScratch(learning_rate=0.1, epochs=5000)

# Entraînement du modèle
custom_model.fit(X_train, y_train)

# Prédiction sur le jeu de test
y_pred_custom = custom_model.predict(X_test)

# Évaluation rapide des performances
accuracy_custom = np.mean(y_pred_custom == y_test)
print(f"\nAccuracy Finale du modèle From Scratch : {accuracy_custom * 100:.2f}%")

Époque 0/5000 - Perte (Loss): 0.6931
Époque 100/5000 - Perte (Loss): 0.6474
Époque 200/5000 - Perte (Loss): 0.6074
Époque 300/5000 - Perte (Loss): 0.5723
Époque 400/5000 - Perte (Loss): 0.5413
Époque 500/5000 - Perte (Loss): 0.5136
Époque 600/5000 - Perte (Loss): 0.4889
Époque 700/5000 - Perte (Loss): 0.4666
Époque 800/5000 - Perte (Loss): 0.4465
Époque 900/5000 - Perte (Loss): 0.4282
Époque 1000/5000 - Perte (Loss): 0.4115
Époque 1100/5000 - Perte (Loss): 0.3962
Époque 1200/5000 - Perte (Loss): 0.3821
Époque 1300/5000 - Perte (Loss): 0.3691
Époque 1400/5000 - Perte (Loss): 0.3571
Époque 1500/5000 - Perte (Loss): 0.3459
Époque 1600/5000 - Perte (Loss): 0.3355
Époque 1700/5000 - Perte (Loss): 0.3258
Époque 1800/5000 - Perte (Loss): 0.3167
Époque 1900/5000 - Perte (Loss): 0.3082
Époque 2000/5000 - Perte (Loss): 0.3001
Époque 2100/5000 - Perte (Loss): 0.2926
Époque 2200/5000 - Perte (Loss): 0.2854
Époque 2300/5000 - Perte (Loss): 0.2787
Époque 2400/5000 - Perte (Loss): 0.2722
Époque 2500/

In [17]:
# Création d'un dictionnaire contenant les paramètres du modèle entraîné
model_artifacts = {
    'weights': custom_model.weights,
    'bias': custom_model.bias
}

# Sauvegarde des paramètres
with open('../src/custom_logistic_model.pkl', 'wb') as f:
    pickle.dump(model_artifacts, f)

print("Poids et biais sauvegardés avec succès!")

Poids et biais sauvegardés avec succès!


## Implémentation Mathématique du Modèle Probabiliste : Naïve Bayes *From Scratch*

En parallèle avec l'approche linéaire, nous implémentons le modèle **Multinomial Naïve Bayes** entièrement *from scratch* avec `NumPy`. Ce modèle est une référence historique et incontournable pour le traitement de texte (NLP) en raison de sa rapidité et de son efficacité face à la haute dimensionnalité.

L'adjectif **"Naïf"** vient de l'hypothèse simplificatrice forte selon laquelle la présence d'un mot dans un message est totalement indépendante de la présence des autres mots, étant donné la classe du message.

---

### A. Fondations Théoriques : Le Théorème de Bayes

Le modèle cherche à calculer la probabilité *a posteriori* qu'un message $D$ (composé des mots $w_1, w_2, \dots, w_n$) appartienne à une classe $c$ (où $c \in \{0: \text{Ham}, 1: \text{Spam}\}$), en utilisant la formule suivante :

$$P(c \mid D) = \frac{P(c) \cdot P(D \mid c)}{P(D)}$$

Puisque le dénominateur $P(D)$ est constant pour toutes les classes lors de l'évaluation d'un même message, nous pouvons l'ignorer et maximiser uniquement le numérateur. En appliquant l'hypothèse d'indépendance naïve, la formule devient :

$$P(c \mid D) \propto P(c) \prod_{i=1}^{n} P(w_i \mid c)$$



---

### Passage au Domaine Logarithmique (Éviter l'Underflow)

En pratique, multiplier des centaines de probabilités très petites (par exemple $0.0001 \times 0.005 \times \dots$) provoque rapidement un **underflow numérique** (le produit s'arrondit à zéro en mémoire). 
Pour résoudre ce problème, on applique la fonction logarithme ($\log$). Les multiplications se transforment alors en sommes, sécurisant ainsi les calculs :

$$\log P(c \mid D) \propto \log P(c) + \sum_{i=1}^{n} \log P(w_i \mid c)$$

Le modèle retient finalement la classe qui obtient le score logarithmique le plus élevé (*Maximum A Posteriori*) :
$$\hat{y} = \arg\max_{c} \left( \log P(c) + \sum_{i=1}^{n} \log P(w_i \mid c) \right)$$

---

### Estimation des Paramètres & Lissage de Laplace

Pendant la phase d'entraînement (`fit`), le modèle calcule deux éléments fondamentaux :

#### Les Probabilités *A Priori* ($\log P(c)$)
C'est la proportion brute de chaque classe dans le jeu d'entraînement :
$$P(c) = \frac{N_c}{N_{\text{total}}}$$
Où $N_c$ est le nombre de documents de la classe $c$.

#### Les Vraisemblances des Mots ($P(w_i \mid c)$) avec Lissage de Laplace
Si un mot du jeu de test n'est jamais apparu dans une classe pendant l'entraînement, sa probabilité $P(w_i \mid c)$ sera égale à $0$. Dans le produit d'origine, cela annulerait toute la probabilité du message. Dans le domaine logarithmique, $\log(0)$ provoquerait une erreur fatale ($-\infty$).

Pour contrer cela, on applique le **lissage de Laplace** (on ajoute $1$ au numérateur et la taille du vocabulaire au dénominateur) :

$$P(w_i \mid c) = \frac{N_{ci} + \alpha}{N_c + \alpha \cdot |V|}$$

* $N_{ci}$ : Somme des scores (ou occurrences) du mot $w_i$ dans la classe $c$.
* $N_c$ : Somme totale de tous les scores de tous les mots dans la classe $c$.
* $|V|$ : Taille totale du vocabulaire global (nombre de caractéristiques, ici 4 385).
* $\alpha$ : Paramètre de lissage (généralement $\alpha = 1$).

---

### Architecture de notre Classe `NaiveBayesFromScratch`
Notre implémentation NumPy repose sur :
1. `fit(X, y)` : Sépare la matrice $X$ par classe, calcule les probabilités *a priori* et applique la formule de Laplace pour générer une matrice de vraisemblance de taille $(2, \text{features})$.
2. `predict(X)` : Applique le produit matriciel dans le domaine logarithmique pour sommer efficacement les poids de chaque mot pour chaque document, puis applique un `np.argmax`.

In [18]:
class NaiveBayesFromScratch:
    def __init__(self, alpha=1.0):
        # alpha = 1.0 correspond au lissage de Laplace standard
        self.alpha = alpha
        self.class_log_prior = None
        self.feature_log_prob = None
        self.classes = None

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.classes = np.unique(y)
        n_classes = len(self.classes)

        # 1. Initialisation des structures pour stocker les statistiques
        # log_prior : Log-probabilité a priori de chaque classe [P(Ham), P(Spam)]
        self.class_log_prior = np.zeros(n_classes)
        # feature_log_prob : Log-vraisemblance de chaque mot pour chaque classe
        self.feature_log_prob = np.zeros((n_classes, n_features))

        # 2. Calcul des probabilités pour chaque classe
        for idx, c in enumerate(self.classes):
            # Extraction des documents appartenant à la classe c
            X_c = X[y == c]
            
            # P(c) = Nombre de documents dans c / Nombre total de documents
            self.class_log_prior[idx] = np.log(X_c.shape[0] / n_samples)

            # 3. Application du lissage de Laplace pour les mots
            # Somme des scores TF-IDF de chaque mot dans la classe c
            count_words_per_feature = np.sum(X_c, axis=0)
            # Somme totale de tous les scores TF-IDF de tous les mots dans la classe c
            total_count_words = np.sum(X_c)

            # Formule : P(w_i | c) = (Somme_mot + alpha) / (Somme_totale_classe + alpha * taille_vocabulaire)
            smoothed_word_probas = (count_words_per_feature + self.alpha) / (total_count_words + self.alpha * n_features)
            
            # Passage dans le domaine logarithmique pour sécuriser les futurs calculs
            self.feature_log_prob[idx, :] = np.log(smoothed_word_probas)

    def predict(self, X):
        # Calcul du score pour chaque document et chaque classe
        # Score = log P(c) + somme(log P(w_i | c))
        # Grâce aux matrices NumPy, on fait ça très efficacement avec un produit matriciel (.T)
        
        predictions = []
        for sample in X:
            # Pour chaque message, on calcule son score pour la classe 0 (Ham) et la classe 1 (Spam)
            class_scores = []
            for idx in range(len(self.classes)):
                prior = self.class_log_prior[idx]
                # Multiplication élément par élément du vecteur TF-IDF avec les log-probabilités apprises
                likelihood = np.sum(sample * self.feature_log_prob[idx])
                class_scores.append(prior + likelihood)
                
            # On choisit l'indice de la classe qui obtient le score maximal (Maximum A Posteriori)
            predictions.append(self.classes[np.argmax(class_scores)])
            
        return np.array(predictions)

In [19]:
# Instanciation du modèle Bayes Naïf From Scratch
nb_custom = NaiveBayesFromScratch(alpha=1.0)

# Entraînement sur les matrices de données vectorisées
nb_custom.fit(X_train, y_train)

# Prédiction sur le jeu de test gardé secret
y_pred_nb = nb_custom.predict(X_test)

# Calcul de l'Accuracy globale
accuracy_nb = np.mean(y_pred_nb == y_test)
print(f"Accuracy du modèle Naïve Bayes From Scratch : {accuracy_nb * 100:.2f}%")

Accuracy du modèle Naïve Bayes From Scratch : 95.42%


In [20]:
# On crée un dictionnaire contenant les connaissances acquises par le modèle probabiliste
naive_bayes_artifacts = {
    'class_log_prior': nb_custom.class_log_prior,
    'feature_log_prob': nb_custom.feature_log_prob,
    'classes': nb_custom.classes
}

# Sauvegarde dans un fichier pickle
with open('../src/custom_naive_bayes_model.pkl', 'wb') as f:
    pickle.dump(naive_bayes_artifacts, f)

print("Paramètres du modèle Naïve Bayes sauvegardés avec succès !")

Paramètres du modèle Naïve Bayes sauvegardés avec succès !
